## Conformational sampling

This part of work prepares necessary files to run MD simulations for colourant and extract conformationw from these runs 

In [1]:
# not too many libraries needed
import os
import parmed as pmd

In [2]:
# Any polyprotic acid sate can be presented in a binary format
# containing 0 and 1 where 0 denotes deprotonated state and 1 - protonated
biliverdin_species = ["S11111", 
                      "S11011",
                      "S10111",
                      "S10011",
                      "S10001",
                      "S00001",
                      "S10000",
                      "S00000"
]

# set up temperatures for parallel tempering 
# efficient transition happens when temperature window is small enough
# wandow size was estimated using this web site: https://virtualchemistry.org/remd-temperature-generator/
temperatures = [str(t) for t in list(range(300, 404, 4))]



In [12]:
# Helper functions definitions

def name2charge(name):
    """Calculates charge of the species"""
    
    return name.count("1") - 4
    
def leap_input(name):
    """
    Leap is a parameterisation program which is a part of Amber suite
    input files are preapred based on the nature of the molecule
    """
    
    charge = name2charge(name)
    # amount of electrolyte ions to keep the NaCl concentration at 0.1 M
    sodium_number = 4 
    chloride_number = 4 
    # update number of ions to make system neutral depdending on the charge
    # of the protonated species
    if charge > 0:
        chloride_number += abs(charge)
    else:
        sodium_number += abs(charge)
    
    leap_inp = f"""
    source leaprc.gaff2
    source leaprc.water.tip3p
    loadamberprep {name}.prepi
    loadamberparams {name}.frcmod
    mol = loadpdb {name}.pdb
    solvatebox mol TIP3PBOX 12
    addions mol Na+ {sodium_number} Cl- {chloride_number}
    saveamberparm mol {name}.prmtop {name}.inpcrd
    quit
    """
    return leap_inp

def min_inp():
    """
    create an input file for Gromacs minimisation
    """
    
    min_mdp = """
    nsteps      = 2000      ; max number of steps
    emtol		= 1  		; convergence total force(kJ/mol/nm) is smaller than
    emstep		= 0.01		; initial step size (nm)
    nstcomm		= 100		; frequency or COM motion removal
    ns_type		= grid
    rlist		= 1.2		; cut-off distance for short range neighbors
    rcoulomb	= 1.2		; distance for coulomb cut-off
    coulombtype	= PME		; electrostatics (Particle Mesh Ewald method)
    fourierspacing	= 0.12		; max grid spacing when using PPPM or PME
    vdw-type	= Shift
    rvdw		= 1.2		; VDW cut-off
    Tcoupl		= no		; temperature coupling
    Pcoupl		= no		; pressure coupling
    gen_vel		= no
    """
    return min_mdp

def equil_inp(t):
    """
    create an input file for Gromacs equilibration run
    argument provides temperature of the simulation
    """
    
    eq_mdp = f"""
    integrator		= md 
    nsteps			= 500000
    dt			    = 0.002
    comm_mode		= Linear 
    nstcomm			= 1000
    nstlog			= 1000
    nstenergy		= 100
    nstxout			= 0
    nstvout			= 0
    nstxtcout       = 1000
    nstfout			= 0
    nstlist			= 10
    ns_type			= grid
    pbc			    = xyz
    rlist			= 1.0 
    rcoulomb		= 1.2
    coulombtype		= pme
    fourierspacing	= 0.12
    vdw-type        = shift
    rvdw            = 1.2
    constraints		= none
    Tcoupl			= v-rescale 
    tc_grps			= system 
    tau_t			= 0.1
    ref_t			= {t}
    Pcoupl			= berendsen
    Pcoupltype		= isotropic
    tau_p			= 1.0
    compressibility	= 4.5e-5
    ref_p			= 1.0 
    gen_vel			= yes
    gen_temp		= {t}
    gen_seed		= -1
    """
    
    return eq_mdp

def prod_inp(t):
    """
    creates an input file for Gromacs production run
    in Replica Exchange mode
    with temperature t
    """
    
    prod_mdp = f"""
    integrator		=  md 
    nsteps			=  25000000
    dt			    =  0.002
    comm_mode		=  Linear 
    nstcomm			=  1000
    nstlog			= 1000
    nstenergy		= 100
    nstxout			= 0
    nstvout			= 0
    nstxtcout       = 5000
    nstfout			= 0
    nstlist			= 10
    ns_type			= grid
    pbc			    = xyz
    rlist			= 1.4 
    rcoulomb		= 1.4
    coulombtype		= pme
    fourierspacing	= 0.12
    vdw-type        = shift
    rvdw            = 1.4 
    constraints		= none
    Tcoupl			= v-rescale
    tc_grps			= system
    tau_t			= 0.1
    ref_t			= {t}
    Pcoupl			= Parrinello-Rahman 
    Pcoupltype		= isotropic
    tau_p			= 1.0
    compressibility	= 4.5e-5
    ref_p			= 1.0 
    gen_vel			= yes
    gen_temp		= {t}
    gen_seed		= -1
    """
    
    return prod_mdp

Preparation steps:
1. Initial coordinates for biliverdin species are prepared in the following way:
  - Atomic coordinates for a fully protonated form are generated
  - Then copied and hydrogen atoms removed to ensure otherwise amber parameterisation program Parmck does not assign a correct MM parameters
2. Antechamber program is used to calculate partial atomic charges
3. Given the prepared parameter set a system for simulation containing water, ions and protonated form is created using tleap
4. Coordinates and parameter files are converted from Amber to Gromacs format
5. Inputs for minimisation, equilibration are prepared and these processes are launched.
6. Input files for the production runs are prepared which can be run on a cluster

In [ ]:
for spec in biliverdin_species:
    os.chdir(f"{spec}")
    
    # charges calculation for a given geometry
    os.system(f"antechamber -i {spec}.pdb -fi pdb -o {spec}.prepi -fo prepi -c bcc -nc {name2charge(spec)} -at gaff")
    
    # parameterisation with General Amber force field
    os.system(f"parmchk2 -i {spec}.prepi -f prepi -o {spec}.frcmod -s gaff")
    
    # create input for a system generation script and run it
    with open("leap.in", "w") as f:
        f.write(leap_input(spec))
    
    os.system(f"tleap -f leap.in")

    #convert amber files to gromacs topologies and coordinates
    parm = pmd.load_file(f"{spec}.prmtop", f"{spec}.inpcrd")
    parm.save(f"{spec}.top") 
    parm.save(f"{spec}.gro") 
    
    # minimise system in gromacs
    with open("min.mdp", "w") as f:
        f.write(min_inp())
    
    os.system(f"gmx_mpi grompp -f min.mdp -c {spec}.gro -p {spec}.top -o em.tpr -maxwarn 200")
    os.system(f"gmx_mpi mdrun -v -deffnm em")
    
    # create a directory for each temperature
    for t in temperatures:
        if not os.path.isdir(t):
            os.mkdir(t)
           
        # create inputs for equilibration    
        os.chdir(f"{t}")
        with open("eq.mdp", "w") as f:
            f.write(equil_inp(t))
        os.system(f"gmx_mpi grompp -f eq.mdp -c ../em.gro -p ../{spec}.top -o eq.tpr -maxwarn 200")
        
        os.chdir("..")
    
    # run equilibration
    os.system(f"mpirun -np 26 --oversubscribe gmx_mpi mdrun -v -multidir [3,4]?? -s eq.tpr")

    # generate inputs for REMD
    for t in temperatures:   
        os.chdir(f"{t}")
        with open("prod.mdp", "w") as f:
            f.write(prod_inp(t))
        os.system(f"gmx_mpi grompp -f prod.mdp -c confout.gro -p ../{spec}.top -o prod.tpr -maxwarn 200")
        os.chdir("..")
        
    # run REMD (uncommmet to run)
    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    # !!!! HIGHLY RECOMMENDED TO RUN THE SIMULATION ON A CLUSTER INSTEAD !!!!
    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    
    #os.system(f"mpirun -np 26 --oversubscribe gmx_mpi mdrun -v -multidir [3,4]?? -s prod.tpr -replex 1000 -reseed 173529")
    
    os.chdir("..")